In [ ]:
prefix_pop = 'hdx'
prefix_adm = 'cbs'

prefix = f'{prefix_pop}_{prefix_adm}'

In [ ]:
import logging

# Create a global logger variable
logger = None

def setupLogger(logFilePath: str) -> None:
    """Sets up a global logger that logs to both console and file, with timestamps.

    Args:
        logFilePath: Path to the log file.
    """
    global logger
    logger = logging.getLogger('dualLogger')
    logger.setLevel(logging.INFO)

    if not logger.handlers:
        formatter = logging.Formatter(
            fmt='%(asctime)s - %(message)s',
            datefmt='%Y-%m-%d %H:%M:%S'
        )

        consoleHandler = logging.StreamHandler()
        consoleHandler.setFormatter(formatter)

        fileHandler = logging.FileHandler(logFilePath, mode='a', encoding='utf-8')
        fileHandler.setFormatter(formatter)

        logger.addHandler(consoleHandler)
        logger.addHandler(fileHandler)

def log(message: str) -> None:
    """Logs a message to both console and file via the global logger.

    Args:
        message: The message to be logged.
    """
    if logger is None:
        raise RuntimeError('Logger has not been initialized. Call setupLogger() first.')
    logger.info(message)

def closeLogger() -> None:
    """Closes and removes all handlers from the global logger."""
    if logger is None:
        return
    for handler in logger.handlers[:]:
        handler.close()
        logger.removeHandler(handler)


In [ ]:
setupLogger('demographics.log')

log(f"Starting{'='*20}{prefix}{'='*20}")

In [ ]:
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Geospatial
import geopandas as gpd

In [ ]:
epsg = 'EPSG:4326' # https://en.wikipedia.org/wiki/World_Geodetic_System#WGS_84

wp_pop = gpd.read_parquet('../Population/wp_nl.parquet').to_crs(epsg)
hdx_pop = gpd.read_parquet('../Population/HDX_NL.parquet').to_crs(epsg)

cbs = gpd.read_parquet('../administrative/cbs.parquet').to_crs(epsg)
gadm = gpd.read_parquet('../administrative/gadm.parquet').to_crs(epsg)

In [ ]:
pop = wp_pop if prefix_pop == 'wp' else hdx_pop
nl = cbs if prefix_adm == 'cbs' else gadm

In [ ]:
assert pop.crs == nl.crs

In [ ]:
log( f"Performing a 'left join within' from {len(pop):,} points" )
res = gpd.sjoin(
    pop, 
    nl, 
    how='left', 
    predicate='within'
).drop(columns=['index_right'])
log( f"resulted in {len(res):,} rows from the {len(pop):,} given rows")

In [ ]:
if len(res.index) > res.index.nunique():
    log( f'{len(res.index) - res.index.nunique():,} duplicate matches found' )
    res = res[~res.index.duplicated(keep='first')]
    log( 'Keeping only the first' )
assert len(res.index) == res.index.nunique()

In [ ]:
assert len(pop) == len(res)

In [ ]:
missing = pop[res.Municipality.isna()]
log( f'Municipality missing for {len(missing):,} out of {len(pop):,} points' )
log( f'{len(missing)/len(pop)*100:.2f}% points')
log( f'{missing.Population.sum()/pop.Population.sum()*100:.2f}% headcount')

In [ ]:
ax = nl.plot(
    color=nl.color_by_city, 
    edgecolor="black", 
    figsize=(13,8), 
    alpha=.5
)  # Plot polygons
missing.plot(
    ax=ax, 
    color="red", 
    markersize=5
)  # Plot missing points
plt.axis('off')
plt.savefig(f'{prefix}_missing.png', format="png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# use a projected CRS (e.g., EPSG:28992 for meters)
projected_crs = 'EPSG:28992'

# Perform nearest join and get the distance
nearest_match = gpd.sjoin_nearest(
    missing.to_crs(projected_crs), 
    nl.to_crs(projected_crs), 
    how="left", 
    distance_col="meters"
)

# Convert back to WGS 84 if needed
nearest_match = nearest_match.to_crs(pop.crs)

In [ ]:
nearest_match.meters.hist()

In [ ]:
res.groupby('Municipality', as_index=True)[['Population', 'Children Under Five',
       'Elderly 60 Plus', 'Men', 'Women', 'Women of Reproductive Age 15 49',
       'Youth 15 24']].sum()

In [ ]:
genders = 'Men' in res.columns

if genders:
    cols_with_counts = [
        'Population', 
        'Children Under Five',
        'Elderly 60 Plus', 
        'Men', 
        'Women', 
        'Women of Reproductive Age 15 49',
        'Youth 15 24'
    ]
    a,b,c = res[['Men','Women','Population']].sum()
    log( f'{(c-a-b)/c:.2f}% people with no gender in the population' )

    groupedDf = res.groupby('Municipality', as_index=True)[
        cols_with_counts
    ].sum()

    a,b,c = groupedDf[['Men','Women','Population']].sum()
    log( f'{(c-a-b)/c:.2f}% people with no gender after grouping' )

In [ ]:
def show_gender_dist(groupedDf, filename):
    # Sort by total population for better visualization
    groupedDf = groupedDf.sort_values(by='Population', ascending=True)

    # Define bar width
    bar_width = 0.4

    # Get index positions for bars
    y_pos = np.arange(len(groupedDf))

    # Create figure and axis with optimized height
    fig, ax = plt.subplots(figsize=(20, min(0.2 * len(groupedDf), 65)))  # Adjust height dynamically

    # Plot Population as a separate bar (green)
    ax.barh(y_pos - bar_width/2, groupedDf['Population'], color='green', label='Total Population',
            height=bar_width, alpha=0.6)

    # Plot stacked bars for Men & Women (blue + red)
    ax.barh(y_pos + bar_width/2, groupedDf['Men'], color='blue', label='Men', height=bar_width)
    ax.barh(y_pos + bar_width/2, groupedDf['Women'], left=groupedDf['Men'],
            color='red', label='Women', height=bar_width)

    # Add annotations for Population bars
    for i, value in enumerate(groupedDf['Population']):
        ax.text(value + 500, y_pos[i] - bar_width/2, f'{int(value):,}', va='center', ha='left', fontsize=8)

    # Add annotations for Men + Women stacked bars
    for i, (men, women) in enumerate(zip(groupedDf['Men'], groupedDf['Women'])):
        total = men + women
        ax.text(total + 500, y_pos[i] + bar_width/2, f'{int(total):,}', va='center', ha='left', fontsize=8)

    # Set y-ticks
    ax.set_yticks(y_pos)
    ax.set_yticklabels(groupedDf.index)

    # Adjust Y-limits to remove whitespace
    ax.set_ylim([-1, len(groupedDf)])

    # Format x-axis ticks with thousands separator
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

    # Add vertical grid lines at x-tick positions
    ax.xaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)

    # Labels and title
    ax.set_xlabel('Population Count')
    ax.set_title('Population Breakdown by Municipality')

    # Add legend
    ax.legend(loc='lower right')

    if filename:
        plt.savefig(f'{prefix}_{filename}.png', format="png", bbox_inches="tight", dpi=300)

    # Show plot
    plt.show()


In [ ]:
if genders:
    show_gender_dist( groupedDf, 'genders' )
    show_gender_dist( groupedDf[groupedDf.Population<=100_000], 'genders_small' )


In [ ]:
groupedDf.columns

In [ ]:
if groupedDf is not None:
    excess_genders = groupedDf[groupedDf.Men + groupedDf.Women > groupedDf.Population].copy()
    excess_genders['Excess'] = (
        excess_genders.Men + excess_genders.Women - excess_genders.Population
    )
    show_gender_dist(
        excess_genders.sort_values('Excess', ascending=False).head(20),
        None
    )

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def plotPieChart(
    data: pd.Series,
    columns: list[str],
    ax: plt.Axes,
    title: str,
    labels: list[str] | None = None,
    colors: list[str] | None = None
) -> None:
    sizes = [data.get(col, 0) for col in columns]
    if labels is None:
        labels = columns

    filtered = [
        (label, size, color if colors else None)
        for label, size, color in zip(labels, sizes, colors or [None] * len(columns))
        if size > 0
    ]

    if not filtered:
        ax.text(0.5, 0.5, 'No data to display', ha='center', va='center', fontsize=12)
        ax.set_title(title)
        ax.axis('off')
        return

    labels, sizes, colors = zip(*filtered)
    ax.pie(
        sizes,
        labels=labels,
        autopct='%1.1f%%',
        startangle=90,
        colors=colors,
        labeldistance=1.05,
        textprops={'fontsize': 9}
    )
    ax.set_title(title, fontsize=12)
    ax.axis('equal')


def plotPopulationBarWithReference(data: pd.Series, ax: plt.Axes, municipality: str) -> None:
    """
    Stacked gender bars, compared with population total as a reference line.
    Annotates Men and Women with % of Population.
    Adds numerical values and delta to the title for self-explanation.
    """
    total = data.get('Population', 0)
    men = data.get('Men', 0)
    women = data.get('Women', 0)
    gender_sum = men + women
    delta = gender_sum - total
    delta_sign = '+' if delta > 0 else ('−' if delta < 0 else '0')

    y_pos = [0]

    # Stacked Men + Women
    ax.barh(y_pos, men, color='skyblue', height=0.4, label='Men')
    ax.barh(y_pos, women, left=men, color='lightcoral', height=0.4, label='Women')

    # Population reference line
    ax.plot([total] * 2, [-0.3, 0.3], color='green', linewidth=2.5, label='Population')

    # Annotations: percentages
    def safe_pct(x):
        return f"{100 * x / total:.1f}%" if total > 0 else "n/a"

    ax.text(men / 2, y_pos[0], safe_pct(men), ha='center', va='center', fontsize=9, color='black')
    ax.text(men + women / 2, y_pos[0], safe_pct(women), ha='center', va='center', fontsize=9, color='black')

    # Axes styling
    ax.set_yticks(y_pos)
    ax.set_yticklabels([''])
    ax.set_xlabel('Population Count')
    ax.set_xlim(0, max(total, gender_sum) * 1.15)
    ax.grid(axis='x', linestyle='--', alpha=0.5)
    ax.legend(loc='lower right', fontsize=8)

    # Compose informative title
    title = (
        # f"Gender vs Population in {municipality}  "
        f"Population: {total:,.1f}, "
        f"Men + Women = {men:,.1f} + {women:,.1f} = {gender_sum:,.1f}, "
        f"Δ = {delta_sign}{abs(delta):,.1f} ({abs(delta)/total*100:,.2f}%)"
    )

    ax.set_title(title, fontsize=12)

def plotMunicipalityPieCharts(grouped_df: pd.DataFrame, municipality: str) -> None:
    """
    Generates demographic plots (1 wide bar, 2 pies) and saves to PDF.
    """
    data = grouped_df.loc[municipality].round(2)

    # Derived values
    children = data.get('Children Under Five', 0)
    youth = data.get('Youth 15 24', 0)
    elderly = data.get('Elderly 60 Plus', 0)
    total = data.get('Population', 0)
    age_sum = children + youth + elderly
    remaining = max(total - age_sum, 0)

    structure_data = data.copy()
    structure_data['Remaining'] = remaining
    structure_data['Other Women'] = max(
        data.get('Women', 0) - data.get('Women of Reproductive Age 15 49', 0), 0
    )

    # Create layout: 2 rows × 2 columns, but span top row over both columns
    fig = plt.figure(figsize=(14, 9))
    fig.suptitle(f'Demographic Breakdown for {municipality}', fontsize=18)

    # Axes placement
    ax_bar = plt.subplot2grid((2, 2), (0, 0), colspan=2)  # spans top row
    ax_pie1 = plt.subplot2grid((2, 2), (1, 0))
    ax_pie2 = plt.subplot2grid((2, 2), (1, 1))

    # Chart 1: Gender vs Population
    plotPopulationBarWithReference(data, ax_bar, municipality)

    # Chart 2: Population structure pie
    plotPieChart(
        structure_data,
        columns=['Children Under Five', 'Youth 15 24', 'Elderly 60 Plus', 'Remaining'],
        ax=ax_pie1,
        title='Population Age Structure',
        labels=['Children <5', 'Youth 15–24', 'Elderly 60+', 'Remaining Ages'],
        colors=['dodgerblue', 'orange', 'forestgreen', 'crimson']
    )

    # Chart 3: Female age structure pie
    plotPieChart(
        structure_data,
        columns=['Women of Reproductive Age 15 49', 'Other Women'],
        ax=ax_pie2,
        title='Female Age Structure',
        colors=['mediumorchid', 'plum']
    )

    # Layout
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    safe_name = municipality.replace(' ', '_')
    pdf_filename = f'{safe_name}_demographics.pdf'
    plt.savefig(pdf_filename, format='pdf', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved figure to {pdf_filename}")


In [ ]:
top_municipality = excess_genders.sort_values('Excess', ascending=False).index[0]
groupedDf.loc[top_municipality].round(2).to_frame()#.to_clipboard()

In [ ]:
[c for c in groupedDf.index if c.startswith('Capelle')]

In [ ]:
plotMunicipalityPieCharts(groupedDf, 'Capelle aan den IJssel')

In [ ]:
plotMunicipalityPieCharts(groupedDf, 'Amsterdam')

In [ ]:
groupedDf.sort_values('Population', ascending=False).head(10)

In [ ]:
for municipality in groupedDf.sort_values('Population', ascending=False).index.values[:10]:
    plotMunicipalityPieCharts(groupedDf, municipality)
    plt.close()  # Close the plot to avoid display

In [ ]:
plotMunicipalityPieCharts(groupedDf, 'Rotterdam')

In [ ]:
if genders:
    men = groupedDf.Men
    women = groupedDf.Women
    population = groupedDf.Population

    ( population - men - women ).hist()

In [ ]:
log(f"Finishing{'='*20}{prefix}{'='*20}")

closeLogger()